# A04 · Hyperparameter search: Kartal (FC128) and Skynet (Conv1x1)

Two figures per network. The **optimization history** shows the score of every trial and the best score so far.
Crosses at the bottom mark trials that the quality gate stopped early, which have no score. The **sensitivity**
plots show the score against each searched parameter (the learning rate of each stage and the entropy
coefficient $c_H$), on a log scale over the whole search range and coloured by trial number. A table compares
the best settings of the two searches.

**Reads** `runs/a2c_hpo/` and `runs/a2c_hpo_1x1/` (T02). **Writes** `figures/a2c_hpo{,_1x1}_history.pdf`,
`figures/a2c_hpo{,_1x1}_sensitivity.pdf` and `figures/a2c_hpo_best.tex`.

In [ ]:
import json
import pathlib
import shutil

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import seaborn as sns

HERE = pathlib.Path.cwd().resolve()  # run from the notebook's folder or from the repository root
EXP = next(p / "experiments" for p in (HERE, *HERE.parents) if (p / "experiments" / "README.md").exists())
RUNS = EXP / "runs"
FIGURES = EXP / "figures"
FIGURES.mkdir(exist_ok=True)

TEXTWIDTH = 16.0 / 2.54
sns.set_theme(context="paper", style="ticks", palette="colorblind", font="serif", rc={
    "text.usetex": shutil.which("latex") is not None,
    "text.latex.preamble": r"\usepackage[T1]{fontenc}\usepackage{amsmath}",
    "font.serif": ["cmr10", "Computer Modern Roman"], "mathtext.fontset": "cm", "axes.formatter.use_mathtext": True,
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9, "legend.fontsize": 8, "legend.title_fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.grid": True, "grid.linewidth": 0.4, "grid.alpha": 0.6,
    "lines.linewidth": 1.1, "figure.dpi": 120, "figure.constrained_layout.use": True,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "pdf.fonttype": 42,
})

## Data

The Skynet run has the suffix `_1x1`. Runs that do not exist yet are skipped.

In [ ]:
SEARCHES = {"FC128": "a2c_hpo", "Conv1x1": "a2c_hpo_1x1"}  # network: run folder, also used as figure name
PARAMETERS = {"learning_rate_collect": "learning rate (collect)", "learning_rate_crates": "learning rate (crates)",
              "learning_rate_versus": "learning rate (versus)", "entropy_coef": r"entropy coefficient $c_H$"}

runs = {}
for network, run in SEARCHES.items():
    if not (RUNS / run / "trials.csv").exists():
        print(f"{network}: no {run}/ yet, skipped")
        continue
    everything = pd.read_csv(RUNS / run / "trials.csv").sort_values("trial")
    trials = everything[everything.state == "COMPLETE"].copy()
    trials["best so far"] = trials.score.cummax()
    if trials.empty:
        print(f"{network}: every trial was stopped by the quality gate, skipped")
        continue
    runs[network] = dict(run=run, config=json.loads((RUNS / run / "config.json").read_text()),
                         best=json.loads((RUNS / run / "best.json").read_text()),
                         trials=trials, pruned=everything[everything.state == "PRUNED"], last=everything.trial.max())
    print(f"{network}: {len(trials)} complete, {len(runs[network]['pruned'])} stopped by the quality gate",
          runs[network]["pruned"].stopped_after.value_counts().to_dict())

## Optimization history

In [ ]:
for network, r in runs.items():
    trials, pruned = r["trials"], r["pruned"]
    fig, ax = plt.subplots(figsize=(TEXTWIDTH, 0.33 * TEXTWIDTH))
    sns.scatterplot(trials, x="trial", y="score", color="0.55", s=12, linewidth=0, label="trial", ax=ax)
    ax.step(trials.trial, trials["best so far"], where="post", color="C0", label="best so far")
    for stage, color in zip(("collect", "crates"), ("C3", "C4")):
        stopped = pruned[pruned.stopped_after == stage]
        if len(stopped):
            ax.scatter(stopped.trial, [0.0] * len(stopped), transform=ax.get_xaxis_transform(), marker="x", s=14,
                       color=color, clip_on=False, label=f"stopped after {stage}")
    ax.set(xlabel="trial", ylabel="score vs. rule-based agent", xlim=(-0.5, r["last"] + 0.5))
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    fig.legend(*ax.get_legend_handles_labels(), loc="outside upper center", ncol=4, frameon=False)
    ax.get_legend().remove()
    fig.savefig(FIGURES / f"{r['run']}_history.pdf")

## Sensitivity

In [ ]:
for network, r in runs.items():
    trials, bounds = r["trials"], r["config"]["bounds"]
    fig, axes = plt.subplots(1, len(PARAMETERS), sharey=True, figsize=(TEXTWIDTH, 0.3 * TEXTWIDTH))
    for ax, (column, label) in zip(axes, PARAMETERS.items()):
        sns.scatterplot(trials, x=column, y="score", hue="trial", palette="viridis", s=14, linewidth=0,
                        legend=False, ax=ax)
        low, high = bounds["entropy_coef" if column == "entropy_coef" else "learning_rate"]
        ax.set(xscale="log", xlabel=label, xlim=(0.8 * low, 1.25 * high))  # the whole search range
        ax.axvline(r["best"][column], color="C3", ls=":", lw=0.8)  # best trial
        ax.xaxis.set_minor_formatter(ticker.NullFormatter())
    axes[0].set_ylabel("score vs. rule-based agent")
    norm = plt.Normalize(trials.trial.min(), trials.trial.max())
    colorbar = fig.colorbar(plt.cm.ScalarMappable(cmap="viridis", norm=norm), ax=axes, pad=0.01, aspect=30)
    colorbar.set_label("trial")
    colorbar.locator = ticker.MaxNLocator(integer=True)
    colorbar.update_ticks()
    fig.savefig(FIGURES / f"{r['run']}_sensitivity.pdf")

## Best points

In [ ]:
def sci(x):
    mantissa, exponent = f"{x:.1e}".split("e")
    return rf"${mantissa}\cdot10^{{{int(exponent)}}}$"


def best_row(r):
    trials, best = r["trials"], r["best"]
    row = trials[trials.trial == best["trial"]].iloc[0]
    return {"trials (complete / gated)": f"{len(trials)} / {len(r['pruned'])}", "best trial": int(best["trial"]),
            **{PARAMETERS[p]: sci(best[p]) for p in PARAMETERS},
            "score": f"{row.score:.2f}", "win rate": f"{row.win:.2f}", "self-kill rate": f"{row.self_kill:.2f}"}


table = pd.DataFrame({network: best_row(r) for network, r in runs.items()})
latex = table.to_latex(escape=False)  # plain \hline, since the report does not use booktabs
for rule in (r"\toprule", r"\midrule", r"\bottomrule"):
    latex = latex.replace(rule, r"\hline")
(FIGURES / "a2c_hpo_best.tex").write_text(latex)
table